# 07. External authz, tool-side (ReBAC)

Pattern 6 gave the tool a cryptographically proven, audience-narrowed
identity for *who is calling*. But the tool still couldn't answer
*relationship* questions about the resources being acted on:

- Bob is a manager. Fine. But can bob approve **alice's** expense
  specifically?
- Alice is in engineering. Fine. But can alice read **this particular**
  document?

The token tells you who's calling. It doesn't tell you the relationship
between the caller and any specific resource. Those questions need a
ReBAC (Relationship-Based Access Control) check, which has to happen
inside the service after the JWT is validated, after the resource is
loaded, with both the caller's identity and the resource's owner in hand.

In this notebook, expense-service calls OPA's `agentauth.tool_side`
package on every `approve_expense` call to ask "can user X approve a
resource owned by user Y?".

## How this notebook differs from pattern 6

Same `TokenExchangeAuth` strategy on the agent side. That part is
identical. The difference is inside expense-service:

```python
# services/expense-service/main.py (excerpt)

if TOOL_SIDE_AUTHZ:
    decision = _opa_tool_side_decision(
        caller=caller, target=target["user_id"],
        action="approve", resource_type="expense",
    )
    if not decision.get("allow"):
        raise HTTPException(status_code=403, detail=f"opa denied: {decision['reason']}")
```

The OPA hook is wired in from the start, gated by an environment variable.
In every previous notebook the env var was off. Notebook 07 turns it on
by recreating the expense-service container with `TOOL_SIDE_AUTHZ=1`.

## Step 1: enable tool-side authz on expense-service

Recreate the expense-service container with the env var flipped on. This
takes 5 to 10 seconds.

In [ ]:
import os
import subprocess
import time

import httpx

REPO_ROOT = os.path.expanduser("~/_develop/exploring-agent-auth")

def restart_expense_service(tool_side_authz: bool) -> None:
    env = os.environ.copy()
    env["TOOL_SIDE_AUTHZ"] = "1" if tool_side_authz else "0"
    subprocess.run(
        ["docker", "compose", "up", "-d", "--no-deps", "--force-recreate", "expense-service"],
        cwd=REPO_ROOT,
        env=env,
        check=True,
        capture_output=True,
    )
    # Wait for /healthz to come back
    for _ in range(30):
        try:
            r = httpx.get("http://localhost:8001/healthz", timeout=1.0)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        time.sleep(0.5)
    raise RuntimeError("expense-service did not become healthy in time")

print("restarting expense-service with TOOL_SIDE_AUTHZ=1...")
status = restart_expense_service(tool_side_authz=True)
print(f"healthz: {status}")

## Step 2: prove the tool-side OPA hook is now active

Recall the policy from `infra/opa/tool_side.rego`:

- Admins (dave) can do anything.
- Users can read their own resources.
- Managers can read AND approve resources owned by their direct reports.
- Same-department peers can read each other's resources (read only).

These rules are evaluated in the *service*, not in the agent. The agent
can ask whatever it wants. The service refuses on its own if the
relationship doesn't permit it.

In [ ]:
from shared.agent import Agent
from shared.auth import TokenExchangeAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

agent = Agent(strategy=TokenExchangeAuth(), tools=ALL_TOOLS)

## Step 3: bob approves alice's expense (manager-of-target relationship)

Bob is alice's manager, per the relationships data loaded into OPA:

```json
"manages": { "bob": ["alice"] }
```

So `tool_side` policy allows bob to approve alice's expense (`expense 4`,
the pending monitor purchase). The agent uses `TokenExchangeAuth`, so the
call carries an exchanged JWT with `sub=bob` and a narrowed `aud`. The
service validates the token, asks OPA whether bob can approve alice's
expense, then proceeds.

In [ ]:
result = run_as("bob", "Approve expense 4, alice's pending monitor purchase.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Step 4: alice tries to approve bob's expense (denied at the tool)

Now flip it. Alice asks the agent to approve **bob's** expense (id 5,
the OAuth workshop). The agent has no agent-side policy preventing this.
It just asks the tool. The tool's OPA hook refuses, because the
`tool_side` policy doesn't permit this relationship.

The error comes back to the LLM, which should explain it to the user
rather than pretend it succeeded.

In [ ]:
result = run_as("alice", "Please approve expense 5, that's bob's training expense.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Step 5: dave (admin) can approve anything

The policy has an admin override: `input.user_id in data.admins`. Dave is
in the admins list, so the OPA decision returns `"reason": "admin override"`.
A useful demo of the layered model: admins are explicitly modeled as a
*data* fact, not a code path, so revoking dave's admin status is a one-line
edit to `infra/opa/data.json` and an OPA reload.

In [ ]:
result = run_as("dave", "Approve expense 5 for me.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Authz lives in *two places* that are intentionally separate.** Pattern
  6's identity propagation answers "who is calling?". Pattern 7's tool-side
  OPA check answers "is this caller permitted to act on this specific
  resource?". They compose: a request must satisfy both.
- **The tool sees a real user**, cryptographically proven, audience-bound,
  AND backed by a fine-grained per-resource decision.
- **Defense in depth.** If the agent has a bug, the tool still enforces
  on its own. If OPA's policy has a bug, the token still has to be valid.
  Multiple layers, each catching a different class of failure.
- **Operational complexity:** higher than pattern 6. The service is now
  responsible for calling OPA on every relevant write, and the OPA
  policies and data files become part of the service's own runtime
  dependencies.
- **Latency:** plus one OPA call per protected action. OPA is fast (sub-ms
  for policies of this size), and easy to run as a sidecar to keep network
  hops local. For chatty services this is usually negligible.
- **Audit trail: the strongest yet.** JWT identifies the caller, OPA
  decision logs say "bob was permitted to approve alice's expense via the
  manager-of-target rule". Auditors can replay every decision against
  historical data and historical policy.

This is the layer most production agentic systems should land on. Once
you have it, the only remaining identity question is whether the *agent*
should have credentials at all.

## Cleanup: turn the tool-side authz off again

So the next notebook (and notebooks 01 to 06 if you re-run them) starts
from a clean default-off state.

In [ ]:
print("restarting expense-service with TOOL_SIDE_AUTHZ=0...")
status = restart_expense_service(tool_side_authz=False)
print(f"healthz: {status}")

## What's still missing

Every pattern from 1 through 7 has the agent holding credentials of some
kind: either a service API key, or knowledge of how to obtain user JWTs
via direct grant. The agent has been the trusted intermediary between
the user and the tools.

What if you don't *want* the agent to be trusted? What if the user should
be able to grant the agent a narrow, time-limited, revocable permission
to call specific tools on their behalf, without the agent ever seeing
their password?

That's what 3-legged OAuth consent flows do. The user goes through a
consent screen in their own browser, the identity provider issues a
token directly to the agent (via the user), and the agent uses that
token to call tools. The user can revoke at any time. The agent never
had the password and can never escalate beyond what the user consented to.

Notebook 08 walks through this. It's interactive: it requires you to
open a URL in your browser and consent.